In [ ]:
import os
from copy import deepcopy
import json
# import xml.etree.ElementTree as ET
# from rich.tree import Tree
# from rich import print as rprint
import io
from typing import List, Union, Tuple, Dict, Optional
from collections.abc import Iterable
from tqdm import tqdm
# import pdfplumber
# import fitz 
import numpy as np
import pandas as pd
import requests
# import xmltodict
import re
from lxml import etree
from pypdf import PdfReader, PdfWriter
from difflib import SequenceMatcher
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.document import DocumentStream
from docling.pipeline.vlm_pipeline import VlmPipeline
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
# device = torch.device("mps")
# from table_link_to_excel import _curl_get_text, _extract_pmc_info, _try_pmc_direct_table_download, _flatten_columns, sanitize_sheet_name, fetch_html, fetch_pmc_fulltext_xml, pick_table, table_to_dataframe, _clean_text
from bs4 import BeautifulSoup
from advp_formatting_engine import *
from advp_information_retriever import *
from advp_table_extraction import *
from utils import *

### Test the pipeline by separating into 2 steps: get the table + get the text col

#### Get the table

In [ ]:
embeddings_model = AutoModel.from_pretrained("NeuML/pubmedbert-base-embeddings")
embeddings_model_tokenizer = AutoTokenizer.from_pretrained("NeuML/pubmedbert-base-embeddings")

In [ ]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC2964649"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
advp_formatting_engine = ADVPFormattingEngine(referencing_col_df)

referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
advp_information_retriever = ADVPInformationRetriever(referencing_col_require_rag_df, use_hf = False, device = "mps")

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    found_table = False
    try:
        has_error = table_link_to_excel(pmid, pmcid)
        if has_error:
            print(f"Error in extracting from {pmid}-{pmcid} with table_link_to_excel")
        else:
            found_table = True
            print(f"Success in extracting from {pmid}-{pmcid} with table_link_to_excel")
    except Exception as e:
        print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    if not found_table:
        try: 
            df_lst = extract_tables_lst_from_paper(pmcid, f"test_papers/{pmid}_{pmcid}.pdf")
            for i, df in enumerate(df_lst):
                if df.shape[0] > 0:
                    df.to_csv(f"intermediate_tables/{pmid}_{pmcid}_{i}_from_pdf.csv", index = False)
                    found_table = True
            if found_table:
                print(f"Success in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
            else:
                print(f"Error in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
        except Exception as e:
            print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
        for file_name in os.listdir("tables"):
            if str(pmid) in file_name and pmcid in file_name:
                if ".xlsx" in file_name:
                    df = pd.read_excel(f"intermediate_tables/{file_name}")
                else:
                    df = pd.read_csv(f"intermediate_tables/{file_name}")

                # save matching dict for debug
                file_name_to_matching[file_name] = advp_formatting_engine.match_many_col_to_ref_col(df)

                harmonized_df = advp_formatting_engine.format_original_table(df, remove_unique_col = True)
                if harmonized_df_all is None:
                    harmonized_df_all = harmonized_df.copy()
                else:
                    harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in harmonizing from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in harmonizing from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
        int_col = ["Chr"]
        for c in int_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
        numerical_col_with_many_numbers = ["Effect"]
        for c in numerical_col_with_many_numbers:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
        float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
        for c in float_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
            if float_col[c] is not None:
                harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in converting to number from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in converting to number from {pmid}-{pmcid} with error {e}")

    # try:
    #     harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
    #     col_require_rag_to_possible_info = advp_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
    # except Exception as e:
    #     print(f"Error in extracting columns from paper text and LLM from {pmid}-{pmcid} with error {e}")
    #     col_require_rag_to_possible_info = {col: [] for col in referencing_col_require_rag_df["column"].unique()}

    # try:
    #     threshold = 0.5
    #     for ref_col, ref_col_choice in zip(
    #         referencing_col_require_rag_with_choice_df["column"], referencing_col_require_rag_with_choice_df["choice"]
    #     ):
    #         if len(col_require_rag_to_possible_info[ref_col]) > 0:
    #             col_with_category = []
    #             if ref_col in col_require_rag_to_possible_info:
    #                 ref_col_choice_lst = ref_col_choice.split(",")
    #                 detail_choice_similarity = calculate_similarity_scores(col_require_rag_to_possible_info[ref_col], ref_col_choice_lst, embeddings_model, embeddings_model_tokenizer)
    #                 # get the max of each col
    #                 max_by_choice = detail_choice_similarity.max(axis = 0).values
    #                 valid_choice = []
    #                 for i in range(len(ref_col_choice_lst)):
    #                     if max_by_choice[i] > threshold:
    #                         valid_choice.append(ref_col_choice_lst[i])
    #                 col_require_rag_to_possible_info[f"{ref_col} category"] = deepcopy(valid_choice)
    #                 col_with_category.append(ref_col)
    #             for ref_col in col_with_category:
    #                 temp, temp_category = col_require_rag_to_possible_info[ref_col], col_require_rag_to_possible_info[f"{ref_col} category"]
    #                 col_require_rag_to_possible_info[ref_col] = deepcopy(temp_category)
    #                 col_require_rag_to_possible_info[f"{ref_col} details"] = deepcopy(temp)
    #                 del col_require_rag_to_possible_info[f"{ref_col} category"]
    #         else:
    #             col_require_rag_to_possible_info[f"{ref_col} details"] = []
    # except Exception as e:
    #     print(f"Error in extracting columns with choice from {pmid}-{pmcid} with error {e}")
    
    # try:
    #     # for cohort need to do differently
    #     # col_require_rag_to_possible_info["Cohort"] = col_require_rag_to_possible_info["Cohort"] + gwas_information_retriever_cohort.extract_possible_info_from_paper(pmid, pmcid)
    #     # col_require_rag_to_possible_info["Cohort"] = list(set([item.lower() for item in col_require_rag_to_possible_info["Cohort"]]))
    #     print(pmid, pmcid)
    #     print(col_require_rag_to_possible_info)
    #     harmonized_df_all = match_possible_info_to_df(harmonized_df_all, col_require_rag_to_possible_info, embeddings_model, embeddings_model_tokenizer)
    #     # harmonized_df_all = match_possible_info_to_df_with_clues(harmonized_df_all, pmid, pmcid, gwas_information_retriever)
    #     # harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
    #     # if (pmid, pmcid) in test_papers_info_sample:
    #     #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}_pred.csv", index = False)
    #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}.csv", index = False)
    #     print(f"Success in extracting columns from paper text from {pmid}-{pmcid}")
    # except Exception as e:
    #     print(f"Error in extracting columns from paper text + matching from {pmid}-{pmcid} with error {e}")

#### Get the text col

In [ ]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC2964649"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
advp_formatting_engine = ADVPFormattingEngine(referencing_col_df)

referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
advp_information_retriever = ADVPInformationRetriever(referencing_col_require_rag_df, use_hf = False, device = "mps")

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    # found_table = False
    # try:
    #     has_error = table_link_to_excel(pmid, pmcid)
    #     if has_error:
    #         print(f"Error in extracting from {pmid}-{pmcid} with table_link_to_excel")
    #     else:
    #         found_table = True
    #         print(f"Success in extracting from {pmid}-{pmcid} with table_link_to_excel")
    # except Exception as e:
    #     print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    # if not found_table:
    #     try: 
    #         df_lst = extract_tables_lst_from_paper(pmcid, f"test_papers/{pmid}_{pmcid}.pdf")
    #         for i, df in enumerate(df_lst):
    #             if df.shape[0] > 0:
    #                 df.to_csv(f"intermediate_tables/{pmid}_{pmcid}_{i}_from_pdf.csv", index = False)
    #                 found_table = True
    #         if found_table:
    #             print(f"Success in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #         else:
    #             print(f"Error in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #     except Exception as e:
    #         print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    # try:
    #     harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
    #     for file_name in os.listdir("tables"):
    #         if str(pmid) in file_name and pmcid in file_name:
    #             if ".xlsx" in file_name:
    #                 df = pd.read_excel(f"intermediate_tables/{file_name}")
    #             else:
    #                 df = pd.read_csv(f"intermediate_tables/{file_name}")

    #             # save matching dict for debug
    #             file_name_to_matching[file_name] = gwas_formatting_engine.match_many_col_to_ref_col(df)

    #             harmonized_df = gwas_formatting_engine.format_original_table(df, remove_unique_col = True)
    #             if harmonized_df_all is None:
    #                 harmonized_df_all = harmonized_df.copy()
    #             else:
    #                 harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
    #     harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
    #     print(f"Success in harmonizing from {pmid}-{pmcid}")
    # except Exception as e:
    #     print(f"Error in harmonizing from {pmid}-{pmcid} with error {e}")

    # try:
    #     harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
    #     harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
    #     int_col = ["Chr"]
    #     for c in int_col:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
    #     numerical_col_with_many_numbers = ["Effect"]
    #     for c in numerical_col_with_many_numbers:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
    #     float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
    #     for c in float_col:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
    #         if float_col[c] is not None:
    #             harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
    #     harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
    #     print(f"Success in converting to number from {pmid}-{pmcid}")
    # except Exception as e:
    #     print(f"Error in converting to number from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        col_require_rag_to_possible_info = advp_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
    except Exception as e:
        print(f"Error in extracting columns from paper text and LLM from {pmid}-{pmcid} with error {e}")
        col_require_rag_to_possible_info = {col: [] for col in referencing_col_require_rag_df["column"].unique()}

    try:
        threshold = 0.5
        for ref_col, ref_col_choice in zip(
            referencing_col_require_rag_with_choice_df["column"], referencing_col_require_rag_with_choice_df["choice"]
        ):
            if len(col_require_rag_to_possible_info[ref_col]) > 0:
                col_with_category = []
                if ref_col in col_require_rag_to_possible_info:
                    ref_col_choice_lst = ref_col_choice.split(",")
                    detail_choice_similarity = calculate_similarity_scores(col_require_rag_to_possible_info[ref_col], ref_col_choice_lst, embeddings_model, embeddings_model_tokenizer)
                    # get the max of each col
                    max_by_choice = detail_choice_similarity.max(axis = 0).values
                    valid_choice = []
                    for i in range(len(ref_col_choice_lst)):
                        if max_by_choice[i] > threshold:
                            valid_choice.append(ref_col_choice_lst[i])
                    col_require_rag_to_possible_info[f"{ref_col} category"] = deepcopy(valid_choice)
                    col_with_category.append(ref_col)
                for ref_col in col_with_category:
                    temp, temp_category = col_require_rag_to_possible_info[ref_col], col_require_rag_to_possible_info[f"{ref_col} category"]
                    col_require_rag_to_possible_info[ref_col] = deepcopy(temp_category)
                    col_require_rag_to_possible_info[f"{ref_col} details"] = deepcopy(temp)
                    del col_require_rag_to_possible_info[f"{ref_col} category"]
            else:
                col_require_rag_to_possible_info[f"{ref_col} details"] = []
    except Exception as e:
        print(f"Error in extracting columns with choice from {pmid}-{pmcid} with error {e}")
    
    try:
        # for cohort need to do differently
        # col_require_rag_to_possible_info["Cohort"] = col_require_rag_to_possible_info["Cohort"] + gwas_information_retriever_cohort.extract_possible_info_from_paper(pmid, pmcid)
        # col_require_rag_to_possible_info["Cohort"] = list(set([item.lower() for item in col_require_rag_to_possible_info["Cohort"]]))
        print(pmid, pmcid)
        print(col_require_rag_to_possible_info)
        harmonized_df_all = match_possible_info_to_df(harmonized_df_all, col_require_rag_to_possible_info, embeddings_model, embeddings_model_tokenizer)
        # harmonized_df_all = match_possible_info_to_df_with_clues(harmonized_df_all, pmid, pmcid, gwas_information_retriever)
        # harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        # if (pmid, pmcid) in test_papers_info_sample:
        #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}_pred.csv", index = False)
        harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in extracting columns from paper text from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in extracting columns from paper text + matching from {pmid}-{pmcid} with error {e}")
# 

In [ ]:
# for f in file_name_to_matching:
#     for ref_col in file_name_to_matching[f]:
#         for i in range(len(file_name_to_matching[f][ref_col])):
#             file_name_to_matching[f][ref_col][i] = (file_name_to_matching[f][ref_col][i][0], float(file_name_to_matching[f][ref_col][i][1]))
# with open("test_matching_dict.json", "w") as f:
#     json.dump(file_name_to_matching, f, indent=4)